# visualization_007

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (220).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `3m` |
| Code cells | 19 |
| Detected functions | preprocess_dataframe, plot_pnl_distribution, plot_pnl_by_exit_reason, plot_trade_summary, plot_total_pnl_by_ticker, plot_pnl_by_weekday, plot_exit_reason_stats, run_visualizations, plot_daily_pnl, monte_carlo_simulation |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Tradebook/merged_output.csv")
df

In [ ]:
# ==========================================
# 📊 Trade Performance Visualization Script
# ==========================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# -----------------------------
# Global Visualization Settings
# -----------------------------
sns.set(style='whitegrid')  # Seaborn aesthetics
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

# -----------------------------
# Data Preprocessing
# -----------------------------
def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and preprocess the input trade DataFrame."""
    df = df.copy()
    df['Entry Time'] = pd.to_datetime(df['Entry Time'], utc=True, format='mixed')
    df['Exit Time'] = pd.to_datetime(df['Exit Time'], utc=True, format='mixed')
    df = df.sort_values('Entry Time')
    df['Exit Reason'] = df['Exit Reason'].astype(str)
    df['Weekday'] = df['Exit Time'].dt.day_name()
    return df

# -----------------------------
# Visualization 1: PnL Histogram
# -----------------------------
def plot_pnl_distribution(df: pd.DataFrame):
    fig = px.histogram(
        df, x='PnL', nbins=300, marginal='rug',
        title='Distribution of PnL',
        opacity=0.75,
        color_discrete_sequence=['forestgreen']
    )
    fig.update_layout(
        xaxis_title='PnL',
        yaxis_title='Frequency',
        bargap=0.05,
        template='plotly_white'
    )
    fig.show()

# --------------------------------------------------
# Visualization 2: PnL Distribution by Exit Reason
# --------------------------------------------------
def plot_pnl_by_exit_reason(df: pd.DataFrame):
    fig = px.histogram(
        df, x='PnL', facet_col='Exit Reason',
        color='Exit Reason', nbins=300, opacity=0.75,
        title='PnL Distribution by Exit Reason',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(
        template='plotly_white',
        showlegend=False,
        height=400
    )
    fig.update_traces(marker_line_width=0.5, marker_line_color='black')
    fig.update_xaxes(matches=None, showticklabels=True)
    fig.update_yaxes(showgrid=True)
    fig.show()

# --------------------------------------------------
# Visualization 3: Trade Summary by Ticker
# --------------------------------------------------
def plot_trade_summary(df: pd.DataFrame):
    df = df.copy()
    df['Win'] = df['PnL'] > 0
    df['Loss'] = df['PnL'] < 0

    summary = df.groupby('Ticker').agg(
        Total_Trades=('Ticker', 'count'),
        Winning_Trades=('Win', 'sum'),
        Losing_Trades=('Loss', 'sum')
    ).reset_index()

    summary_melted = summary.melt(
        id_vars='Ticker',
        value_vars=['Winning_Trades', 'Losing_Trades'],
        var_name='Trade Type',
        value_name='Count'
    )

    fig = px.bar(
        summary_melted,
        x='Ticker', y='Count', color='Trade Type',
        barmode='group', text_auto=True,
        title='Trade Summary by Ticker',
        labels={'Count': 'Number of Trades', 'Ticker': 'Stock Ticker'},
        color_discrete_map={'Winning_Trades': 'green', 'Losing_Trades': 'red'}
    )
    fig.update_layout(
        xaxis_tickangle=-45,
        title_font_size=16,
        xaxis_title_font_size=14,
        yaxis_title_font_size=14,
        legend_title_text='Trade Outcome'
    )
    fig.show()

# --------------------------------------------------
# Visualization 4: Total PnL by Ticker
# --------------------------------------------------
def plot_total_pnl_by_ticker(df: pd.DataFrame):
    pnl_by_ticker = df.groupby('Ticker')['PnL'].sum().sort_values(ascending=False)
    plt.figure(figsize=(15, 6))
    pnl_by_ticker.plot(kind='bar', color='mediumpurple', edgecolor='black')
    plt.title('Total PnL by Ticker')
    plt.xlabel('Ticker')
    plt.ylabel('Total PnL')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# --------------------------------------------------
# Visualization 5: Total PnL by Weekday
# --------------------------------------------------
def plot_pnl_by_weekday(df: pd.DataFrame):
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    weekday_pnl = df.groupby('Weekday')['PnL'].sum().reset_index()
    weekday_pnl['Weekday'] = pd.Categorical(weekday_pnl['Weekday'], categories=weekday_order, ordered=True)
    weekday_pnl = weekday_pnl.sort_values('Weekday')
    weekday_pnl['PnL_Type'] = weekday_pnl['PnL'].apply(lambda x: 'Positive' if x > 0 else 'Negative')

    fig = px.bar(
        weekday_pnl, x='Weekday', y='PnL', text_auto=True,
        title='Total PnL by Weekday',
        labels={'PnL': 'Total PnL', 'Weekday': 'Day of Week'},
        color='PnL_Type',
        color_discrete_map={'Positive': 'green', 'Negative': 'red'}
    )
    fig.update_layout(
        title_font_size=16,
        xaxis_title_font_size=14,
        yaxis_title_font_size=14,
        legend_title_text='PnL Type'
    )
    fig.show()

# --------------------------------------------------
# Visualization 6: Exit Reason Analysis
# --------------------------------------------------
def plot_exit_reason_stats(df: pd.DataFrame):
    exit_stats = df.groupby('Exit Reason')['PnL'].agg(['sum', 'mean', 'count']).reset_index()
    exit_stats.rename(columns={'sum': 'Total PnL', 'mean': 'Average PnL', 'count': 'Number of Trades'}, inplace=True)

    print(exit_stats)

    # Total PnL per Exit Reason
    plt.figure(figsize=(10, 5))
    sns.barplot(data=exit_stats, x='Exit Reason', y='Total PnL', palette='viridis')
    plt.title('Total PnL by Exit Reason')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Average PnL per Exit Reason
    plt.figure(figsize=(10, 5))
    sns.barplot(data=exit_stats, x='Exit Reason', y='Average PnL', palette='magma')
    plt.title('Average PnL by Exit Reason')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# -----------------------------
# Master Runner
# -----------------------------
def run_visualizations(df: pd.DataFrame):
    df = preprocess_dataframe(df)
    plot_pnl_distribution(df)
    plot_pnl_by_exit_reason(df)
    plot_trade_summary(df)
    plot_total_pnl_by_ticker(df)
    plot_pnl_by_weekday(df)
    plot_exit_reason_stats(df)

# Group by Ticker and Direction
grouped = df.groupby(['Ticker', 'Direction'])['PnL'].sum().unstack(fill_value=0)

# Plot
ax = grouped.plot(kind='bar', figsize=(12,6))

plt.xlabel("Ticker")
plt.ylabel("Cumulative PnL")
plt.title("Cumulative Long vs Short PnL per Ticker")
plt.legend(title="Direction")
plt.grid(axis='y', linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# =============================
# 🚀 Run All Visualizations
# =============================https://accounts.google.com/SignOutOptions?hl=en&continue=https://colab.research.google.com/drive/1Huv_lsL5aXHdXneFAdMSSU8aqGQ9lJRU&ec=GBRAqQM
run_visualizations(df)


In [ ]:
summary = df['Exit Reason'].value_counts().reset_index()
summary.columns = ['Exit Reason', 'Count']
print("Number of unique exit reasons:", len(summary))
print(summary)

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Assuming 'df' has already been sorted and includes 'PnL'
df = df.sort_values('Entry Time')
df['Cumulative PnL'] = df['PnL'].cumsum()
df['Peak'] = df['Cumulative PnL'].cummax()
df['Drawdown'] = ((df['Cumulative PnL'] - df['Peak']) / 500000) * 100# As percent

# Create figure with subplots
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=("Portfolio Balance Over Time", "Drawdown (%)")
)

# 1. Balance Over Time
fig.add_trace(
    go.Scatter(x=df['Entry Time'], y=df['Cumulative PnL'], mode='lines', name='Balance', line=dict(color='purple')),
    row=1, col=1
)

# 2. Drawdown Percentage
fig.add_trace(
    go.Scatter(x=df['Entry Time'], y=df['Drawdown'], mode='lines', name='Drawdown %', line=dict(color='red')),
    row=2, col=1
)

# Update layout
fig.update_layout(
    height=700,
    width=1000,
    title_text="Portfolio Performance Overview (Interactive)",
    showlegend=True,
    template='plotly_white'
)

# Axis labels
fig.update_yaxes(title_text="Balance", row=1, col=1)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=1)
fig.update_xaxes(title_text="Entry Time", row=2, col=1)

fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

def plot_daily_pnl(df, initial_capital=100000):
    """
    Plot daily PnL % vs initial capital
    - Green bars = profit days
    - Red bars = loss days
    """
    # Ensure datetime
    df['Entry Time'] = pd.to_datetime(df['Entry Time'])

    # Group by date and sum daily PnL
    daily_pnl = df.groupby(df['Entry Time'].dt.date).agg({'PnL': 'sum'}).reset_index()
    daily_pnl.rename(columns={'Entry Time': 'Date'}, inplace=True)

    # Convert to % of initial capital
    daily_pnl['Daily %'] = (daily_pnl['PnL'] / initial_capital) * 100

    # Chart
    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=daily_pnl['Date'],
            y=daily_pnl['Daily %'],
            marker_color=daily_pnl['Daily %'].apply(lambda x: 'green' if x >= 0 else 'red'),
            opacity=0.8
        )
    )

    # Reference line at 0
    fig.add_hline(y=0, line_color="black")
    fig.add_hline(y=-3, line_color="black")

    fig.update_layout(
        height=800,
        width=1400,
        title_text=f"Daily PnL % of Initial Capital (${initial_capital:,.0f})",
        yaxis_title="Daily PnL (%)",
        xaxis_title="Date",
        template="plotly_white"
    )

    return fig, daily_pnl

# Usage
fig, dd_data = plot_daily_pnl(df, initial_capital=100000)
fig.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# === Prepare Data ===
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['YearMonth'] = df['Entry Time'].dt.to_period('M')
df['PnL %'] = (df['PnL'] / 100000) * 100

pnl_pct_matrix = df.groupby(['Ticker', 'YearMonth'])['PnL %'].sum().unstack().fillna(0)
pnl_pct_matrix.columns = pnl_pct_matrix.columns.astype(str)

# Split into two parts: first 36 months, then the remaining
month_columns = pnl_pct_matrix.columns.tolist()
first_part = pnl_pct_matrix[month_columns[:36]]
second_part = pnl_pct_matrix[month_columns[36:]]

# Color map (centered at 0)
cmap = mcolors.LinearSegmentedColormap.from_list("red_white_green", ["red", "white", "green"])
vmax = abs(pnl_pct_matrix.values).max()

# === Plotting ===
fig, axes = plt.subplots(2, 1, figsize=(24, 10), dpi=300)  # Wide and High DPI

# --- Top heatmap (First 36 Months) ---
sns.heatmap(
    first_part,
    ax=axes[0],
    annot=True,
    fmt=".1f",
    cmap=cmap,
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=0.3,
    linecolor='gray',
    annot_kws={"size": 14},
    cbar=False
)
axes[0].set_title("Monthly PnL %", fontsize=22, pad=10)
axes[0].set_xlabel("")
axes[0].set_ylabel("Ticker", fontsize=16)
axes[0].tick_params(axis='x', rotation=90, labelsize=12)
axes[0].tick_params(axis='y', labelsize=14)

# --- Bottom heatmap (Last 18 Months) ---
sns.heatmap(
    second_part,
    ax=axes[1],
    annot=True,
    fmt=".1f",
    cmap=cmap,
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=0.3,
    linecolor='gray',
    annot_kws={"size": 14},
    cbar_kws={"shrink": 0.6}
)
axes[1].set_title("Monthly PnL %", fontsize=22, pad=10)
axes[1].set_xlabel("Month", fontsize=16)
axes[1].set_ylabel("Ticker", fontsize=16)
axes[1].tick_params(axis='x', rotation=90, labelsize=12)
axes[1].tick_params(axis='y', labelsize=14)

# === Final Layout and Save ===
plt.tight_layout(h_pad=4)
plt.savefig("split_monthly_pnl_heatmap_highres.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors

# === Prepare Data ===
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['YearMonth'] = df['Entry Time'].dt.to_period('M')
df['PnL %'] = (df['PnL'] / 100000) * 100

pnl_pct_matrix = df.groupby(['Ticker', 'YearMonth'])['PnL %'].sum().unstack().fillna(0)
pnl_pct_matrix.columns = pnl_pct_matrix.columns.astype(str)

# ✅ Take only the first 8 months
first_8 = pnl_pct_matrix.iloc[:, :32]

# Color map (centered at 0)
cmap = mcolors.LinearSegmentedColormap.from_list("red_white_green", ["red", "white", "green"])
vmax = abs(first_8.values).max()

# === Plotting ===
plt.figure(figsize=(14, 6), dpi=300)
sns.heatmap(
    first_8,
    annot=True,
    fmt=".1f",
    cmap=cmap,
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=0.3,
    linecolor='gray',
    annot_kws={"size": 14},
    cbar_kws={"shrink": 0.6}
)

plt.title("Monthly PnL % (First 9 Months)", fontsize=22, pad=10)
plt.xlabel("Month", fontsize=16)
plt.ylabel("Ticker", fontsize=16)
plt.xticks(rotation=90, fontsize=12)
plt.yticks(rotation=45, fontsize=14)  # 🔄 Rotate y-axis labels

plt.tight_layout()
plt.savefig("first_8_months_pnl_heatmap.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Monthly return % based on ₹1L capital
df['YearMonth'] = df['Entry Time'].dt.to_period('M')
df['PnL %'] = (df['PnL'] / 100000) * 100

# Group by Ticker and Month, sum PnL % for monthly returns
monthly_returns = df.groupby(['Ticker', 'YearMonth'])['PnL %'].sum().reset_index()

# Pivot: Ticker x Month
returns_matrix = monthly_returns.pivot(index='Ticker', columns='YearMonth', values='PnL %').sort_index(axis=1)

# Compute rolling Sharpe (mean / std over 3 months)
rolling_mean = returns_matrix.rolling(window=3, axis=1).mean()
rolling_std = returns_matrix.rolling(window=3, axis=1).std()
rolling_sharpe = rolling_mean / rolling_std

rolling_sharpe.to_csv("rolling_3M_sharpe_ratio.csv")

# Plot heatmap
plt.figure(figsize=(50, 5))
sns.heatmap(rolling_sharpe, annot=True, fmt=".2f", cmap='RdYlGn', center=0,
            linewidths=0.5, linecolor='gray', cbar_kws={'label': 'Rolling 3M Sharpe'}, annot_kws={"size": 10})
plt.title('Rolling 3-Month Sharpe Ratio per Ticker', fontsize=16, pad=12)
plt.xlabel('Month')
plt.ylabel('Ticker')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Convert 'Entry Time' to datetime and extract the hour
df['Hour'] = pd.to_datetime(df['Entry Time']).dt.hour

# Categorize trades as Win or Loss
df['Trade Type'] = df['PnL'].apply(lambda x: 'Win' if x > 0 else 'Loss')

# Group by hour to get total PnL per hour
pnl_per_hour = df.groupby('Hour')['PnL'].sum().reset_index()

# Create subplot figure
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=("Number of Win & Loss Trades Per Hour", "Total PnL Per Hour"))

# Histogram for trade count
histogram = px.histogram(df, x='Hour', color='Trade Type', barmode='group')

# Bar chart for total PnL per hour
pnl_bar = go.Bar(x=pnl_per_hour['Hour'], y=pnl_per_hour['PnL'], name='Total PnL', marker_color='blue')

# Add traces to the subplot
for trace in histogram.data:
    fig.add_trace(trace, row=1, col=1)

fig.add_trace(pnl_bar, row=2, col=1)

# Update layout
fig.update_layout(title_text="Trade Analysis Per Hour", xaxis_title="Hour of Day", height=700, width= 1250)

# Show figure
fig.show()


In [ ]:
import pandas as pd

# Prepare data
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
pnl_matrix = df.pivot_table(index='Entry Time', columns='Ticker', values='PnL')
traded = pnl_matrix.notnull() & (pnl_matrix != 0)

num_traded_tickers = traded.sum(axis=1)
total_times = len(num_traded_tickers)

percentages = {}
for i in range(traded.shape[1], 0, -1):  # from max tickers down to 1
    count_i = (num_traded_tickers == i).sum()
    percentages[i] = (count_i / total_times) * 100

# Print output in requested format
for n in range(traded.shape[1], 0, -1):
    pct = percentages[n]
    print(f"{pct:.2f}% of time all {n} tickers are traded at the same time")


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import calendar

# Ensure 'Entry Time' is in datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Extract month and year
df['Year'] = df['Entry Time'].dt.year
df['Month'] = df['Entry Time'].dt.month

# Monthly total profit percent
monthly_profit = df.groupby(['Year', 'Month'])['PnL'].sum().reset_index()
monthly_profit['Profit Percent'] = (monthly_profit['PnL'] / 500000) * 100

# Yearly return
yearly_return = df.groupby('Year')['PnL'].sum().reset_index()
yearly_return['Profit Percent'] = (yearly_return['PnL'] / 500000) * 100

# Pivot table for heatmap
pivot_table = monthly_profit.pivot(index='Year', columns='Month', values='Profit Percent')
pivot_table.columns = [calendar.month_abbr[i] for i in pivot_table.columns]

# 📊 Heatmap
heatmap = px.imshow(
    pivot_table,
    text_auto=".2f",
    color_continuous_scale="YlGnBu",
    labels={"color": "Profit %"},
    title="Monthly Total Profit Percentage",
)

heatmap.update_layout(
    width=1200,
    height=500,
    xaxis_title="Month",
    yaxis_title="Year",
    xaxis=dict(tickmode="array", tickvals=list(range(len(pivot_table.columns))), ticktext=list(pivot_table.columns)),
    yaxis=dict(tickmode="array", tickvals=pivot_table.index),
    coloraxis_colorbar=dict(title="Profit %"),
)

heatmap.show()

# 📊 Bar Chart for Yearly Return
bar_chart = go.Figure()

bar_chart.add_trace(go.Bar(
    x=yearly_return['Year'],
    y=yearly_return['Profit Percent'],
    marker=dict(color="skyblue"),
    text=yearly_return['Profit Percent'].round(2),
    textposition='outside',
))

bar_chart.update_layout(
    width=1200,
    height=500,
    title="Yearly Total Profit Percentage",
    xaxis_title="Year",
    yaxis_title="Yearly Return (%)",
    bargap=0.2,
    template="plotly_white"
)

bar_chart.show()


In [ ]:
import pandas as pd
import numpy as np

# === Assuming df is already loaded ===
# Convert Entry/Exit Time to datetime if not already
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['Exit Time'] = pd.to_datetime(df['Exit Time'])

# === DAILY SHARPE CALCULATION ===
df['Trade Date'] = df['Entry Time'].dt.date
daily_pnl = df.groupby('Trade Date')['PnL'].sum()

mean_daily_return = daily_pnl.mean()
std_daily_return = daily_pnl.std()
sharpe_daily = (mean_daily_return / std_daily_return) * np.sqrt(365) if std_daily_return != 0 else np.nan

# === BASIC METRICS ===
total_trades = len(df)
winning_trades = df[df['PnL'] > 0]
losing_trades = df[df['PnL'] < 0]
win_rate = len(winning_trades) / total_trades
loss_rate = len(losing_trades) / total_trades
avg_win = winning_trades['PnL'].mean()
avg_loss = losing_trades['PnL'].mean()
avg_pnl = df['PnL'].mean()
total_pnl = df['PnL'].sum()
profit_factor = winning_trades['PnL'].sum() / abs(losing_trades['PnL'].sum()) if len(losing_trades) > 0 else np.inf
expectancy = (win_rate * avg_win) + (loss_rate * avg_loss)

# === CUMULATIVE PNL & DRAWDOWN ===
df['Cumulative PnL'] = df['PnL'].cumsum()
cumulative_pnl = df['Cumulative PnL']
running_max = cumulative_pnl.cummax()
drawdown = running_max - cumulative_pnl
max_drawdown = drawdown.max()

# === Drawdown Duration ===
drawdown_periods = []
start = None

for i in range(len(cumulative_pnl)):
    if cumulative_pnl[i] < running_max[i]:
        if start is None:
            start = i
    else:
        if start is not None:
            duration = (df.iloc[i]['Entry Time'] - df.iloc[start]['Entry Time']).total_seconds() / 60
            drawdown_periods.append(duration)
            start = None
if start is not None:
    duration = (df.iloc[-1]['Exit Time'] - df.iloc[start]['Entry Time']).total_seconds() / 60
    drawdown_periods.append(duration)

max_drawdown_duration = max(drawdown_periods) if drawdown_periods else 0
avg_drawdown_duration = np.mean(drawdown_periods) if drawdown_periods else 0

# === Sortino Ratio ===
downside_returns = daily_pnl[daily_pnl < 0]
downside_std = downside_returns.std() if len(downside_returns) > 0 else np.nan
sortino_ratio = (mean_daily_return / downside_std) * np.sqrt(365) if downside_std != 0 else np.nan

# === Calmar Ratio ===
annualized_return = mean_daily_return * 365
calmar_ratio = annualized_return / max_drawdown if max_drawdown != 0 else np.nan

# === Holding Time ===
df['Holding Time'] = (df['Exit Time'] - df['Entry Time']).dt.total_seconds() / 60
avg_holding_time = df['Holding Time'].mean()

# === Total Time Span ===
start_time = df['Entry Time'].min()
end_time = df['Exit Time'].max()
total_time_span = (end_time - start_time).total_seconds() / 60  # in minutes

# === TRUE TIME INVESTED WITHOUT OVERLAP ===
intervals = list(zip(df['Entry Time'], df['Exit Time']))
merged_intervals = []

for start, end in sorted(intervals):
    if not merged_intervals or start > merged_intervals[-1][1]:
        merged_intervals.append([start, end])
    else:
        merged_intervals[-1][1] = max(merged_intervals[-1][1], end)

true_time_invested = sum([(end - start).total_seconds() / 60 for start, end in merged_intervals])  # in minutes
true_investment_ratio = true_time_invested / total_time_span if total_time_span != 0 else np.nan

# === Average Number of Trades Per Day ===
unique_days = df['Trade Date'].nunique()
avg_trades_per_day = total_trades / unique_days if unique_days != 0 else np.nan

# === OUTPUT METRICS ===
metrics = {
    "Total Trades": total_trades,
    "Win Rate": win_rate,
    "Loss Rate": loss_rate,
    "Average PnL": avg_pnl,
    "Total PnL": total_pnl,
    "Profit Factor": profit_factor,
    "Expectancy": expectancy,
    "Max Drawdown": max_drawdown,
    "Sharpe Ratio (Daily)": sharpe_daily,
    "Sortino Ratio": sortino_ratio,
    "Calmar Ratio": calmar_ratio,
    "Average Holding Time (mins)": avg_holding_time,
    "True Time Invested (mins)": true_time_invested,
    "Total Time Span (mins)": total_time_span,
    "True Investment Ratio": true_investment_ratio,
    "Max Drawdown Duration (mins)": max_drawdown_duration,
    "Avg Drawdown Duration (mins)": avg_drawdown_duration,
    "Average Trades Per Day": avg_trades_per_day
}

# === Print the Metrics ===
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")


In [ ]:
# 📂 Ensure datetime conversion
df['Entry Time'] = pd.to_datetime(df['Entry Time'])
df['Exit Time'] = pd.to_datetime(df['Exit Time'])

# 🛠 Create Exit Date
df['Exit Date'] = df['Exit Time'].dt.date

# 🛠 Daily Aggregation PER TICKER
daily = df.groupby(['Ticker', 'Exit Date']).agg({
    'PnL': 'sum',
    'Entry Price': 'mean',
    'Exit Price': ['max', 'min']
})
daily.columns = ['PnL', 'Entry Price', 'Max Exit Price', 'Min Exit Price']

# 📈 Calculate Daily Range % (volatility proxy)
daily['Daily Range %'] = (daily['Max Exit Price'] - daily['Min Exit Price']) / daily['Entry Price'] * 100

# 🔥 Define Volatility Clusters PER TICKER
daily['Volatility Regime'] = daily.groupby('Ticker')['Daily Range %'] \
                                  .transform(lambda x: np.where(x > x.median(), 'High Volatility', 'Low Volatility'))

# 🔗 Merge Volatility Regime back to df
df = df.merge(
    daily[['Volatility Regime']],
    left_on=['Ticker', df['Exit Time'].dt.date],
    right_index=True
)

# 📊 Analyze Performance by Ticker + Volatility
volatility_performance = df.groupby(['Ticker', 'Volatility Regime']).agg({
    'PnL': ['mean', 'sum', 'count'],
    'Direction': 'count'
})
volatility_performance.columns = ['Avg PnL', 'Total PnL', 'Number of Trades', 'Number of Trades Direction']

# 🎯 Win Rate calculation
win_rate = (
    df.assign(Win=df['PnL'] > 0)
      .groupby(['Ticker', 'Volatility Regime'])['Win']
      .mean() * 100
)

# 🎨 Plotting - Example: for one ticker at a time
ticker = df['Ticker'].unique()[0]  # pick first ticker
volatility_performance.loc[ticker]['Avg PnL'].plot(
    kind='bar', title=f'Average PnL per Trade by Volatility Regime - {ticker}',
    color=['steelblue', 'salmon']
)
plt.ylabel('Average PnL (₹)')
plt.show()

win_rate.loc[ticker].plot(
    kind='bar', title=f'Win Rate by Volatility Regime - {ticker}',
    color=['seagreen', 'tomato']
)
plt.ylabel('Win Rate (%)')
plt.show()

# 🖨 Print the table
print(volatility_performance)
print("\nWin Rates (%):")
print(win_rate)


In [ ]:
# 📊 Collapse into combined summary (equal weight for each ticker)
combined = volatility_performance.groupby('Volatility Regime').mean(numeric_only=True)

# 🎯 Win Rate combined (equal-weight across tickers)
combined_win_rate = win_rate.groupby('Volatility Regime').mean()

# 🎨 Plotting - same style as before
fig, axes = plt.subplots(2, 1, figsize=(5, 7))  # 2 rows, 1 column

# 📈 Top: Average PnL per Trade
combined['Avg PnL'].plot(
    kind='bar',
    ax=axes[0],
    title='Average PnL per Trade (Combined across Tickers)',
    color=['steelblue', 'salmon']
)
axes[0].set_ylabel('Average PnL (₹)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', labelrotation=0)
axes[0].grid(False)  # 🔴 Remove grid

# 📈 Bottom: Win Rate
combined_win_rate.plot(
    kind='bar',
    ax=axes[1],
    title='Win Rate (Combined across Tickers)',
    color=['seagreen', 'tomato']
)
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('Volatility Regime')
axes[1].tick_params(axis='x', labelrotation=0)
axes[1].grid(False)  # 🔴 Remove grid

plt.tight_layout()
plt.show()

# 🖨 Print the combined tables
print("Combined Performance (Average across 5 tickers):")
print(combined)
print("\nCombined Win Rate (%):")
print(combined_win_rate)


In [ ]:
import pandas as pd
import plotly.express as px

# Ensure datetime format
df['Entry Time'] = pd.to_datetime(df['Entry Time'])

# Pivot: rows = Entry Time, columns = Ticker, values = PnL
pnl_matrix = df.pivot_table(index='Entry Time', columns='Ticker', values='PnL')

# Drop columns with very few non-null values to clean matrix
pnl_matrix = pnl_matrix.dropna(thresh=10, axis=1)

# Fill missing values (optional but useful)
pnl_matrix = pnl_matrix.fillna(0)

# Compute correlation matrix
pnl_corr = pnl_matrix.corr()

# Plot correlation heatmap using Plotly
fig = px.imshow(
    pnl_corr,
    title='PnL Correlation Between Tickers (Aligned by Entry Time)',
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1
)
fig.update_layout(width=900, height=900)
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load your tradebook data
# Example: df = pd.read_csv("tradebook.csv")

# Step 1: Ensure 'Exit Time' is datetime and extract date
df["Exit Time"] = pd.to_datetime(df["Exit Time"])
df["Date"] = df["Exit Time"].dt.date

# Step 2: Aggregate daily PnL and normalize by fixed ₹10 lakh capital
daily_pnl = df.groupby("Date")["PnL"].sum()
daily_returns = daily_pnl / 5_00_000  # Normalized return per ₹10 lakh

# Step 3: Monte Carlo Simulation Function
def monte_carlo_simulation(pnl_series, num_simulations=5_00_000, num_days=None):
    if num_days is None:
        num_days = len(pnl_series)

    # Sample daily returns with replacement
    simulations = np.random.choice(pnl_series, size=(num_simulations, num_days), replace=True)
    cumulative_returns = np.cumsum(simulations, axis=1)
    return cumulative_returns

# Step 4: Run the simulation
simulations = monte_carlo_simulation(daily_returns, num_simulations=5_00_000)

# Step 5: Plot first 500 simulation paths
plt.figure(figsize=(15, 6))
for sim in simulations[:500]:  # Corrected to 500
    plt.plot(sim, color="gray", alpha=0.1)
plt.title("Monte Carlo Simulations of Strategy PnL (500 paths)")
plt.xlabel("Days")
plt.ylabel("Cumulative Return (Normalized to ₹10L)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 6: Print Summary Statistics
final_returns = simulations[:, -1]
print("Mean Final Return:", round(np.mean(final_returns), 2))
print("5th Percentile Return (Worst Case):", round(np.percentile(final_returns, 5), 2))
print("95th Percentile Return (Best Case):", round(np.percentile(final_returns, 95), 2))
print("Minimum Return:", round(np.min(final_returns), 2))
print("Maximum Return:", round(np.max(final_returns), 2))


In [ ]:
plt.hist(final_returns, bins=100, color="skyblue", edgecolor="black")
plt.title("Distribution of Final Returns")
plt.xlabel("Final Return (Normalized)")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()
